# Data Analytics Competition (DAC) Find IT! 2026
## Face Anti-Spoofing Detection
This notebook provides a complete pipeline for training a state-of-the-art vision model to classify face images into 6 categories (1 real, 5 fake) and generate a submission file.

In [ ]:
!pip install -q timm albumentations scikit-learn pandas matplotlib tqdm

In [ ]:
import os
import random
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

## Configuration
We use EfficientNet-B3 as it offers a strong trade-off between accuracy and speed, along with hyperparameters configured for image classification.

In [ ]:
class CFG:
    seed = 42
    model_name = 'tf_efficientnet_b3.ns_jft_in1k' # High performance pre-trained efficientnet
    img_size = 300
    epochs = 10
    batch_size = 32
    lr = 1e-4
    min_lr = 1e-6
    weight_decay = 1e-2
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    n_folds = 5
    # Classes defined by the competition rules
    classes = ['realperson', 'fake_printed', 'fake_screen', 'fake_mask', 'fake_mannequin', 'fake_unknown']
    num_classes = len(classes)

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(CFG.seed)
print(f"Using device: {CFG.device}")

## Data Loading and Preparation
Download dataset using kaggle CLI before running this part. Assuming the data is stored in the `data/` directory (`train/` and `test/`).

In [ ]:
from pathlib import Path

DATA_DIR = Path('data') # adjust if your data is extracted elsewhere
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
SAMPLE_SUB = DATA_DIR / 'sample_submission.csv'

# Read training images dynamically from directories
train_data = []
if TRAIN_DIR.exists():
    for class_idx, class_name in enumerate(CFG.classes):
        class_dir = TRAIN_DIR / class_name
        if class_dir.exists():
            for img_path in class_dir.glob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    train_data.append({
                        'image_path': str(img_path),
                        'label': class_idx,
                        'class_name': class_name
                    })

df = pd.DataFrame(train_data)
if len(df) > 0:
    print(f'Total training images: {len(df)}')
    print("\nClass Distribution:")
    print(df['class_name'].value_counts())
else:
    print('Warning: No training data found. Please run kaggle competitions download and unzip.')

## Dataset and Augmentations
Albumentations is used to provide advanced image transformations to increase dataset variety and model robustness.

In [ ]:
class FaceSpoofDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        self.image_paths = df['image_path'].values
        self.labels = df['label'].values if 'label' in df.columns else None
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        # Handle images gracefully
        try:
            image = np.array(Image.open(img_path).convert('RGB'))
        except Exception as e:
            print(f"Error reading {img_path}: {e}")
            image = np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8)
        
        if self.transform:
            image = self.transform(image=image)['image']
            
        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            return image, label
        return image

def get_transforms(data='train'):
    if data == 'train':
        return A.Compose([
            A.Resize(CFG.img_size, CFG.img_size),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
            A.GaussNoise(p=0.1),
            A.Normalize(
                mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225], 
                max_pixel_value=255.0
            ),
            ToTensorV2()
        ])
    elif data == 'valid':
        return A.Compose([
            A.Resize(CFG.img_size, CFG.img_size),
            A.Normalize(
                mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225], 
                max_pixel_value=255.0
            ),
            ToTensorV2()
        ])

## Model Architecture
We use `timm` to load a pre-trained EfficientNet model and modify the classification head for 6 classes.

In [ ]:
class FaceModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained)
        
        # Dynamically change the final layer for different architectures
        if 'efficientnet' in model_name:
            in_features = self.model.classifier.in_features
            self.model.classifier = nn.Linear(in_features, num_classes)
        elif 'convnext' in model_name:
            in_features = self.model.head.fc.in_features
            self.model.head.fc = nn.Linear(in_features, num_classes)
        elif 'resnet' in model_name or 'resnext' in model_name:
            in_features = self.model.fc.in_features
            self.model.fc = nn.Linear(in_features, num_classes)
        else:
            self.model.reset_classifier(num_classes) # fallback
            
    def forward(self, x):
        return self.model(x)

## Training and Validation Functions

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    epoch_loss = 0.0
    preds, true_labels = [], []
    
    pbar = tqdm(dataloader, desc='Train', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * images.size(0)
        preds.append(outputs.argmax(1).cpu().numpy())
        true_labels.append(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    preds = np.concatenate(preds)
    true_labels = np.concatenate(true_labels)
    macro_f1 = f1_score(true_labels, preds, average='macro')
    
    return epoch_loss / len(dataloader.dataset), macro_f1

def valid_epoch(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0.0
    preds, true_labels = [], []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Valid', leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            epoch_loss += loss.item() * images.size(0)
            preds.append(outputs.argmax(1).cpu().numpy())
            true_labels.append(labels.cpu().numpy())
            
    preds = np.concatenate(preds)
    true_labels = np.concatenate(true_labels)
    macro_f1 = f1_score(true_labels, preds, average='macro')
    
    return epoch_loss / len(dataloader.dataset), macro_f1

## Local Cross Validation Loop (Training)
Using StratifiedKFold to handle potential class imbalances effectively.

In [ ]:
# Execute this block only if data is correctly loaded
if len(df) > 0:
    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    df['fold'] = -1
    for fold, (_, val_idx) in enumerate(skf.split(df, df['label'])):
        df.loc[val_idx, 'fold'] = fold
        
    # We demonstrate training only Fold 0 for speed.
    fold = 0
    print(f'========== Fold {fold} ==========')
    
    train_df = df[df['fold'] != fold].reset_index(drop=True)
    valid_df = df[df['fold'] == fold].reset_index(drop=True)
    
    train_dataset = FaceSpoofDataset(train_df, transform=get_transforms('train'))
    valid_dataset = FaceSpoofDataset(valid_df, transform=get_transforms('valid'))
    
    train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    model = FaceModel(CFG.model_name, CFG.num_classes)
    model.to(CFG.device)
    
    # CrossEntropyLoss deals well with multi-class classification
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs, eta_min=CFG.min_lr)
    
    best_f1 = 0.0
    
    for epoch in range(CFG.epochs):
        print(f'Epoch {epoch+1}/{CFG.epochs}')
        
        train_loss, train_f1 = train_epoch(model, train_loader, optimizer, criterion, CFG.device)
        valid_loss, valid_f1 = valid_epoch(model, valid_loader, criterion, CFG.device)
        
        scheduler.step()
        
        print(f'Train Loss: {train_loss:.4f} | Train Macro F1: {train_f1:.4f}')
        print(f'Valid Loss: {valid_loss:.4f} | Valid Macro F1: {valid_f1:.4f}')
        
        if valid_f1 > best_f1:
            print(f'Validation Macro F1 improved from {best_f1:.4f} to {valid_f1:.4f}. Saving model...')
            best_f1 = valid_f1
            torch.save(model.state_dict(), f'{CFG.model_name}_fold{fold}_best.pth')
        print('-' * 40)

## Inference and Submission
Load the best weights from training and evaluate the unlabelled test data, formatting it to Kaggle requirements.

In [ ]:
def get_test_images():
    test_data = []
    if TEST_DIR.exists():
        for img_path in TEST_DIR.glob('*'):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                id_name = img_path.stem # Extracts 'test_001' from 'test_001.jpg'
                test_data.append({
                    'id': id_name,
                    'image_path': str(img_path)
                })
    return pd.DataFrame(test_data)

test_df = get_test_images()
if len(test_df) > 0:
    # Sort by ID just to be organized
    test_df = test_df.sort_values('id').reset_index(drop=True)
    test_dataset = FaceSpoofDataset(test_df, transform=get_transforms('valid'))
    test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size*2, shuffle=False, num_workers=2)
    
    model = FaceModel(CFG.model_name, CFG.num_classes, pretrained=False)
    # Load best model weights
    model_path = f'{CFG.model_name}_fold0_best.pth'
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path))
        print(f"Loaded model weights from {model_path}")
    else:
        print("Warning: Model weights not found. Running with uninitialized weights!")
        
    model.to(CFG.device)
    model.eval()
    
    predictions = []
    with torch.no_grad():
        for images in tqdm(test_loader, desc='Inference'):
            images = images.to(CFG.device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            predictions.extend(preds)
            
    # Map numerical predictions back to string labels needed for submission format
    pred_labels = [CFG.classes[p] for p in predictions]
    
    submission = pd.DataFrame({
        'id': test_df['id'],
        'label': pred_labels
    })
    
    submission.to_csv('submission.csv', index=False)
    print('\nSubmission saved to submission.csv')
    print(submission.head())
else:
    print('No test data found. Make sure test images are inside the "data/test/" directory.')
    if SAMPLE_SUB.exists():
        # Produce dummy submission if data not found, so Kaggle scoring works on notebooks
        sub_df = pd.read_csv(SAMPLE_SUB)
        sub_df['label'] = 'realperson' 
        sub_df.to_csv('submission.csv', index=False)
        print('Dummy submission saved to submission.csv as fail-safe.')
